In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print('Libraries loaded')

print("Loading data")
cols = [
'dti',
'fico_range_low',
'inq_last_6mths',
'int_rate',
'issue_d',
'loan_amnt',
'loan_status',
'pub_rec',
'revol_util',
'sub_grade',
'term'
]
dfa_data = pd.read_parquet('LCA_cleaned_final', columns=cols) #2,260,701 

dlqcy = ['Charged Off','Late (31-120 days)','Late (16-30 days)','Default']
dflt = ['Charged Off','Default']

dfa_data['delinquency_tf'] = dfa_data['loan_status'].isin(dlqcy).astype(int)
dfa_data['defaulted_tf'] = dfa_data['loan_status'].isin(dflt).astype(int)

dfa = dfa_data.sample(n=500000, random_state=42) #Will contain the columns above, plus the new columns for delinquency and default. 

print("Complete")

Libraries loaded
Loading data
Complete


In [2]:
# Target is 'defaulted_tf' or 'delinquency_tf'

default_predictors = ['sub_grade','revol_util','inq_last_6mths', 'loan_amnt', 'int_rate','term']   # your chosen predictors

X = dfa[default_predictors].copy()   # Predictor matrix
y = dfa['defaulted_tf']              # Target vector

In [3]:
X = pd.get_dummies(X, columns=['sub_grade'], drop_first=True) # One-hot encode the 'sub_grade' categorical into a matrix of binary features. 
#Note - Can only run this once, as it will create new columns in X. If you run it again, it will throw an error because the columns already exist.

In [4]:
print(X.head(8).to_string())

                     revol_util  inq_last_6mths  loan_amnt  int_rate  term  sub_grade_A2  sub_grade_A3  sub_grade_A4  sub_grade_A5  sub_grade_B1  sub_grade_B2  sub_grade_B3  sub_grade_B4  sub_grade_B5  sub_grade_C1  sub_grade_C2  sub_grade_C3  sub_grade_C4  sub_grade_C5  sub_grade_D1  sub_grade_D2  sub_grade_D3  sub_grade_D4  sub_grade_D5  sub_grade_E1  sub_grade_E2  sub_grade_E3  sub_grade_E4  sub_grade_E5  sub_grade_F1  sub_grade_F2  sub_grade_F3  sub_grade_F4  sub_grade_F5  sub_grade_G1  sub_grade_G2  sub_grade_G3  sub_grade_G4  sub_grade_G5
__null_dask_index__                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                    # X and y will be split into training and testing sets
    test_size=0.3,           # 30% held out for testing
    random_state=42,         # Makes results reproducible across runs
    stratify=y               # Preserve class balance (important for rare events like default)
)

In [6]:
model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1,
    class_weight='balanced'  # Adjust weights inversely proportional to class frequencies                
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]   # Probability of positive class (default)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.58      0.71    132165
           1       0.19      0.72      0.30     17835

    accuracy                           0.59    150000
   macro avg       0.56      0.65      0.51    150000
weighted avg       0.85      0.59      0.67    150000


ROC-AUC Score: 0.7108


In [7]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Features:")
print(importance.head(15))


Top Features:
           feature  importance
3         int_rate    0.494871
1   inq_last_6mths    0.075463
0       revol_util    0.068457
4             term    0.058506
2        loan_amnt    0.050318
7     sub_grade_A4    0.035159
5     sub_grade_A2    0.033981
6     sub_grade_A3    0.028887
8     sub_grade_A5    0.019978
9     sub_grade_B1    0.017488
10    sub_grade_B2    0.014118
14    sub_grade_C1    0.012624
13    sub_grade_B5    0.011752
11    sub_grade_B3    0.009704
12    sub_grade_B4    0.007597


In [8]:
#Run a confusion matrix to see how many of each class were predicted correctly and incorrectly.
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

"""Confusion Matrix:
[[76197 55968]     >>> True positives defaults 76197, False positives defaults 55968
[ 4911 12924]]     >>> False negatives non-defaults 4911, True negatives non-defaults 12924
 
Performance proportionally is at 0.59414 correct from a 150k sample.  

 """



Confusion Matrix:
[[76197 55968]
 [ 4911 12924]]


'Confusion Matrix:\n[[76197 55968]     >>> True positives defaults 76197, False positives defaults 55968\n[ 4911 12924]]     >>> False negatives non-defaults 4911, True negatives non-defaults 12924\n\nPerformance proportionally is at 0.59414 correct from a 150k sample.  \n\n '

## Model updates:

- 7/9/2026: First pass failed. 0 prediction for Class 1 defaults using 30% of sample. (150k)
    - pass 1 precision, recall, f1-score all 0.00. 
    Classification Report:
                precision    recall  f1-score   support

            0       0.88      1.00      0.94    132165
            1       0.00      0.00      0.00     17835

    accuracy                            0.88    150000
    macro avg       0.44      0.50      0.47    150000
    weighted avg    0.78      0.88      0.83    150000
    ROC-AUC Score: 0.71

    - pass 2 huge improvement. Changed class_weight to 'balanced'. 
        Classification Report:
                    precision    recall  f1-score   support

                0       0.94      0.58      0.71    132165
                1       0.19      0.72      0.30     17835

        accuracy                            0.59    150000
        macro avg       0.56      0.65      0.51    150000
        weighted avg    0.85      0.59      0.67    150000
        ROC-AUC Score: 0.7108


# Thoughts:
[[76197 55968]     >>> True positives defaults 76197, False positives defaults 55968
[ 4911 12924]]     >>> False negatives non-defaults 4911, True negatives non-defaults 12924

- From pass 2, and the confusion matrix; even though there are technically “false positives” relative to what actually happened, Lending Club still failed as a company. So maybe some of what our model is flagging as risky should have been treated as risky. At the end of the day Lending Club’s decision to approve them was part of why they struggled. In other words, blindly treating Lending Club’s historical approvals as “correct” might mean we’re learning their mistakes instead of improving on them.

In our FP cohort, the absolute highest sub_grade is B4. About 45% of the sub grades in that FP cohort are D1 or worse. And that's only in our ~55k of False Positives cohort. A very small sample of the entire data set. 
- Average revol_util is 57.14112494. 
- Average inquiry is 0.802726558, so HIGHLY likely to have an inquiry. 
- Sum of loans granted - $872,143,250.00 in loans given out.
- Average interest rates for FP is ~17%.  

So in the False Positive cohort, it is **absolutely NOT the case** that the majority of the loans were 'performing just fine'. The breakdown of that cohort is that they simply did not hit the specific 'default' or 'charged off' status. 

In [ ]:
results = X_test.copy()
results['actual'] = y_test.values
results['predicted'] = y_pred
results['proba_default'] = y_pred_proba

# Create category column
def get_category(row):
    if row['actual'] == 0 and row['predicted'] == 0:
        return 'TN'
    elif row['actual'] == 0 and row['predicted'] == 1:
        return 'FP'
    elif row['actual'] == 1 and row['predicted'] == 0:
        return 'FN'
    else:
        return 'TP'

results['category'] = results.apply(get_category, axis=1)

print("Results dataframe created successfully.")
print(results['category'].value_counts())


results[results['category'] == 'TP'].to_excel(r'C:\Users\spenc\Downloads\True_Positives.xlsx', index=False)
results[results['category'] == 'FP'].to_excel(r'C:\Users\spenc\Downloads\False_Positives.xlsx', index=False)
results[results['category'] == 'FN'].to_excel(r'C:\Users\spenc\Downloads\False_Negatives.xlsx', index=False)
results[results['category'] == 'TN'].to_excel(r'C:\Users\spenc\Downloads\True_Negatives.xlsx', index=False)
print("\n Files exported")

In [ ]:
# =============================================================================
# Template for a basic machine learning workflow in Python
# =============================================================================


"""
STEP 1: Load the cleaned data
- Use the parquet file from File 1 (much faster than CSV)
- In finance projects, always prefer Parquet for large datasets
"""


"""
STEP 2: Define target and features
- Target = binary outcome we want to predict (default / delinquency)
- Features = the variables we selected after EDA, VIF, and importance checks
- Repeatable tip: Keep a clear list here so you can easily swap models later
"""
target = '<your_target_column>'          # e.g. 'defaulted_tf' or 'delinquency_tf'

features = ['<column1>', '<column2>', '<column3>']   # your chosen predictors

y = dfa[target]            # Target vector. A series of 0s and 1s that the model is learning to predict. 
X = dfa[features].copy()   # Predictor matrix. A pandas DataFrame containing the features used to predict the above target.


"""
STEP 3: Train / Test Split
- Always split before training so you can honestly test generalization
- stratify=y keeps the proportion of good/bad loans similar in both sets
- In finance, this is critical because class imbalance (few defaults) is common
"""
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,           # 30% held out for testing
    random_state=42,         # Makes results reproducible across runs
    stratify=y               # Preserve class balance (important for rare events like default)
)

"""
STEP 4: Train the model
- Random Forest is a good starting point for tabular finance data
- n_estimators = number of trees (more = more stable but slower)
- max_depth = prevents trees from growing too deep (helps control overfitting)
"""
model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1                # Use all CPU cores for speed
)
model.fit(X_train, y_train)

"""
STEP 5: Evaluate performance
- classification_report gives precision, recall, F1 per class
- ROC-AUC is especially useful in finance (measures ability to rank risk)
"""
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]   # Probability of positive class (default)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))

"""
STEP 6: Feature Importance
- This shows which variables the model actually relied on most
- Compare this to your earlier correlation / VIF results
"""
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Features:")
print(importance.head(15))

In [ ]:
# =============================================================================
# Template 2 for a more complex machine learning workflow in Python
# =============================================================================

"""
Notes for grok later:
How might we handle class imbalance? (e.g., SMOTE, class weights)
How might we tune hyperparameters? (e.g., GridSearchCV, RandomizedSearchCV
What do I need to examine to determine class_weight? (e.g., class distribution, ROC-AUC, precision-recall curves)
What is target leakage?
"Censored / Survival Targets" - in real lending some loans haven't matured yet, so we don't know if they will default. How do we handle that?
    - possibly with 'survival analysis' or 'time-to-event' models. 
Default can mean different things across lenders. For example, some lenders consider a loan defaulted after 90 days of non-payment, while others may use 120 days. Make sure to check the definition in your dataset.
How can we create multi-class targets? Maybe number risk 1-5, or use a regression model to predict probability of default.
How do we deal with 'lagged' targets? For example, if we want to predict default in the next 12 months, we need to make sure our features are from before that time period. Otherwise, we might be using future information to predict the past (target leakage).
"""

## Ideas I want to remember later... Or when working on more complex problems:

### Categorical Feature Engineering Strategy: Thematic Grouping + One-Hot Encoding
**Strategy:**  
Instead of blindly one-hot encoding every unique value in a high-cardinality column (like `purpose`), group similar categories together based on business meaning or keyword patterns (e.g., anything related to "car", "truck", "auto", "RV", "motorcycle" → `purpose_vehicle`), then one-hot encode the reduced set of meaningful groups.

- Reduces dimensionality compared to full one-hot encoding.
- Creates more interpretable and business-relevant features.
- Allows me to test specific themes (vehicles, debt consolidation, home improvement, etc.) rather than noisy individual values.

AI says - This is sometimes called **"Semantic Grouping"**, **"Keyword-Based Bucketing"**, or **"Domain-Driven Feature Aggregation"** before encoding.